# 보기 좋은 그래프로

> 파이썬 10강 · 시각화

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [보기 좋은 그래프로](https://mioon1402.github.io/timeseriesdata/python/p10-plot-polish.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# 표를 글자로 찍을 때 한글 열이 어긋나지 않게 (한글을 두 칸으로 계산)
import pandas as pd
pd.set_option("display.unicode.east_asian_width", True)

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 날짜 축이 겹칠 때

**10-1. 문제와 해결**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.5))

# 문제 — 그냥 그리면 라벨이 겹친다
ax1.plot(df["date"], df["visitors"], linewidth=0.6)
ax1.set_title("✗ 날짜 라벨이 겹친다")

# 해결 — 간격과 형식을 지정한다
ax2.plot(df["date"], df["visitors"], linewidth=0.6, color="#2563eb")
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))   # 3개월마다
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))   # 표시 형식
ax2.set_title("○ 3개월 간격 + 형식 지정")

fig.autofmt_xdate()      # 라벨을 살짝 기울여준다
plt.tight_layout()
plt.show()

## 2. 숫자 축 읽기 쉽게

**10-2. 축 숫자 다듬기**

In [ ]:
from matplotlib.ticker import FuncFormatter

월별 = df.set_index("date")["sales"].resample("ME").mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))

# 문제 — 자릿수가 많아 읽기 어렵다
ax1.plot(월별.index, 월별.values)
ax1.set_title("✗ 원 단위 그대로")

# 해결 1 — 단위를 바꾼다 (가장 좋은 방법)
ax2.plot(월별.index, 월별.values / 10000, color="#0d9488")
ax2.set_ylabel("매출(만원)")
ax2.set_title("○ 만원 단위로")

plt.tight_layout()
plt.show()

**10-3. 천 단위 쉼표 넣기**

In [ ]:
from matplotlib.ticker import FuncFormatter

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(월별.index, 월별.values, color="#2563eb")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: f"{x:,.0f}"))
ax.set_title("천 단위 쉼표로 읽기 쉽게")
ax.set_ylabel("매출(원)")
plt.show()

## 3. 색은 의미가 있을 때만

**10-4. 강조하고 싶은 것만 색으로**

In [ ]:
순서 = ["월", "화", "수", "목", "금", "토", "일"]
요일평균 = df.groupby("weekday")["visitors"].mean().reindex(순서)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))

# 문제 — 무지개색. 색이 아무 정보도 주지 않는다
ax1.bar(요일평균.index, 요일평균.values,
        color=["red", "orange", "yellow", "green", "blue", "navy", "purple"])
ax1.set_ylim(0, 240)
ax1.set_title("✗ 색이 의미 없음")

# 해결 — 기본은 회색, 강조할 것만 색
색 = ["#cbd5e1"] * 7
색[5] = "#2563eb"     # 토요일만 강조
ax2.bar(요일평균.index, 요일평균.values, color=색)
ax2.set_ylim(0, 240)
ax2.set_title("○ 토요일이 가장 높다는 메시지")

plt.tight_layout()
plt.show()

## 4. 범례와 주석

**10-5. 여러 선에는 범례를**

In [ ]:
df["연"] = df["date"].dt.year
df["월"] = df["date"].dt.month
표 = df.pivot_table(values="visitors", index="월", columns="연", aggfunc="mean")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(표.index, 표[2024], marker="o", label="2024년", color="#94a3b8")
ax.plot(표.index, 표[2025], marker="o", label="2025년", color="#2563eb")

ax.set_xlabel("월")
ax.set_ylabel("평균 방문객(명)")
ax.set_title("월별 방문객 — 2년 비교")
ax.legend()
ax.set_xticks(range(1, 13))
plt.show()

**10-6. 주석으로 사건 표시하기**

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.plot(df["date"], df["visitors"], linewidth=0.6, color="#94a3b8")

# 방송 대박 날을 찾아서 화살표로 표시
대박 = df.loc[df["visitors"].idxmax()]
ax.annotate(
    f"방송 노출\n{대박['visitors']}명",
    xy=(대박["date"], 대박["visitors"]),           # 가리킬 지점
    xytext=(대박["date"], 대박["visitors"] + 60),  # 글자 위치
    ha="center", fontsize=10, color="#dc2626",
    arrowprops=dict(arrowstyle="->", color="#dc2626"),
)

ax.set_title("이상치에 설명을 붙이면 그래프가 스스로 말한다")
ax.set_ylabel("방문객(명)")
plt.show()

## 5. 군더더기 걷어내기

**10-7. 테두리와 격자 정리**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))

ax1.bar(요일평균.index, 요일평균.values, color="#2563eb")
ax1.set_ylim(0, 240)
ax1.grid(True)
ax1.set_title("✗ 격자가 데이터를 가린다")

ax2.bar(요일평균.index, 요일평균.values, color="#2563eb")
ax2.set_ylim(0, 240)
ax2.spines["top"].set_visible(False)      # 위 테두리 제거
ax2.spines["right"].set_visible(False)    # 오른쪽 테두리 제거
ax2.grid(axis="y", alpha=0.3)             # 가로 격자만, 연하게
ax2.set_axisbelow(True)                   # 격자를 막대 뒤로
ax2.set_title("○ 필요한 것만 남김")

plt.tight_layout()
plt.show()

## 6. 완성본 만들기

**10-8. 완성본**

In [ ]:
import matplotlib.dates as mdates

월매출 = df.set_index("date")["sales"].resample("ME").mean() / 10000

fig, ax = plt.subplots(figsize=(9.5, 4))

ax.plot(월매출.index, 월매출.values, color="#2563eb", linewidth=2, marker="o", markersize=4)
ax.fill_between(월매출.index, 월매출.values, alpha=0.12, color="#2563eb")

# 축 정리
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%y년 %m월"))
ax.set_ylabel("월평균 일매출 (만원)")
ax.set_ylim(0, 130)

# 군더더기 제거
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)

# 메시지가 담긴 제목
성장 = (월매출.iloc[-6:].mean() / 월매출.iloc[:6].mean() - 1) * 100
ax.set_title(f"매출이 2년간 꾸준히 성장 (최근 6개월이 초기 대비 +{성장:.0f}%)",
             fontsize=13, fontweight="bold", loc="left", pad=14)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 7. 저장하기

**10-9. 파일로 저장**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(요일평균.index, 요일평균.values, color="#2563eb")
ax.set_ylim(0, 240)
ax.set_title("요일별 평균 방문객")

# 브라우저 안에서는 가상 파일시스템에 저장된다
fig.savefig("요일별.png", dpi=150, bbox_inches="tight")
print("저장했습니다")

import os
print("파일 크기:", os.path.getsize("요일별.png"), "바이트")
plt.show()

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 10-5 의 2년 비교 그래프를 다듬어보세요.
#        - 위/오른쪽 테두리 제거
#        - 가로 격자만 연하게
#        - 제목에 '메시지'를 담기


# 문제 2. 기온과 매출의 산점도를 그리되,
#        비 온 날은 다른 색으로 표시하고 범례를 붙여보세요.
#        힌트: 두 번 scatter 를 호출하고 label 을 주기

**모범 답안**

In [ ]:
# 문제 1
fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.plot(표.index, 표[2024], marker="o", label="2024년", color="#cbd5e1")
ax.plot(표.index, 표[2025], marker="o", label="2025년", color="#2563eb")
ax.set_xticks(range(1, 13)); ax.set_xlabel("월"); ax.set_ylabel("평균 방문객(명)")
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
증가 = (표[2025].mean() / 표[2024].mean() - 1) * 100
ax.set_title(f"모든 달에서 성장 (연평균 +{증가:.0f}%)",
             fontweight="bold", loc="left")
ax.legend(frameon=False)
plt.show()

# 문제 2
비 = df["rain_mm"] > 0
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(df.loc[~비, "avg_temp"], df.loc[~비, "sales"] / 10000,
           s=10, alpha=0.35, color="#94a3b8", label="비 안 온 날")
ax.scatter(df.loc[비, "avg_temp"], df.loc[비, "sales"] / 10000,
           s=14, alpha=0.7, color="#2563eb", label="비 온 날")
ax.set_xlabel("평균기온(℃)"); ax.set_ylabel("매출(만원)")
ax.set_title("비 오는 날은 같은 기온에서도 매출이 낮다", fontweight="bold", loc="left")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
plt.show()

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)